In [1]:
import torch 
import torch.nn as nn
import tiktoken
import torch_directml

In [2]:
device = torch_directml.device()

In [3]:
print("Using device:", device)

Using device: privateuseone:0


In [4]:
with open("wizardofoz.txt","r",encoding="utf-8") as f:
    data = f.read()

## TOO BIG FOR MY GPU
GPT_CONFIG_124M = {
   "vocab_size" : 50257,
    "context_length" : 1024,
    "emb_dim" : 768,
    "n_heads" : 12,
    "n_layers" : 12,
    "drop_rate" : 0.1,
    "qkv_bias" : False
}

In [5]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 128,
    "emb_dim": 256,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [6]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
       super().__init__()
       self.eps = eps
       self.scale = nn.Parameter(torch.ones(emb_dim))
       self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
       mean = x.mean(dim=-1, keepdim=True)
       variance = x.var(dim=-1, keepdim=True, unbiased=False)
       normalized = (x - mean) / torch.sqrt(variance + self.eps)
       return self.scale * normalized + self.shift


In [7]:
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device, dtype=x.dtype))
                * (x + 0.044715 * x.pow(3))
            )
        )

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        emb_dim = cfg["emb_dim"]
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim),
        )

    def forward(self, x):
        return self.layers(x)

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False
    ):
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out must be divisible by num_heads")
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        causal_mask = torch.triu(
            torch.ones(context_length, context_length, dtype=torch.bool),
            diagonal=1
        )
        self.register_buffer("causal_mask", causal_mask)
    def forward(self, x):
        batch_size, num_tokens, _ = x.shape
        queries = self.query(x)
        keys = self.key(x)
        values = self.value(x)
        queries = queries.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        keys = keys.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        values = values.view(
            batch_size, num_tokens, self.num_heads, self.head_dim
        ).transpose(1, 2)
        scores = queries @ keys.transpose(2, 3)
        mask = self.causal_mask[:num_tokens, :num_tokens]
        scores = scores.masked_fill(mask, -torch.inf)
        weights = torch.softmax(scores / (self.head_dim ** 0.5), dim=-1)
        weights = self.dropout(weights)
        context = (weights @ values).transpose(1, 2).contiguous()
        context = context.view(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.feed_forward = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.residual_dropout = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.residual_dropout(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.residual_dropout(x)
        x = x + shortcut
        return x

In [10]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.context_length = cfg["context_length"]
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.position_embedding = nn.Embedding(
            cfg["context_length"], cfg["emb_dim"]
        )
        self.embedding_dropout = nn.Dropout(cfg["drop_rate"])
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.output_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError(
                f"Input has {sequence_length} tokens; maximum is {self.context_length}."
            )
        token_embeddings = self.token_embedding(token_ids)
        position_ids = torch.arange(sequence_length, device=token_ids.device)
        position_embeddings = self.position_embedding(position_ids)
        x = token_embeddings + position_embeddings
        x = self.embedding_dropout(x)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        return self.output_head(x)

In [11]:
tokenizer = tiktoken.get_encoding("gpt2")

In [12]:
encoded = tokenizer.encode(data[:10000])

context_length = GPT_CONFIG_124M["context_length"]

input_batch = torch.tensor(
    [encoded[:context_length]],
    dtype=torch.long,
    device=device
)

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M).to(device)

logits = model(input_batch)

print("Input batch shape:", tuple(input_batch.shape))
print("Logits shape:", tuple(logits.shape))

Input batch shape: (1, 128)
Logits shape: (1, 128, 50257)


In [13]:
def report_model_size(model):
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    parameter_bytes = sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    )
    print(f"Total parameters: {total_parameters:,}")
    print(f"Parameter storage: {parameter_bytes / (1024** 2):.2f} MiB")
    return total_parameters

report_model_size(model)

Total parameters: 28,920,832
Parameter storage: 110.32 MiB


28920832

In [15]:
def generate_text(model, token_ids, max_new_tokens, context_size):
    model.eval()
    for _ in range(max_new_tokens):
        model_input = token_ids[:, -context_size:]
        with torch.no_grad():
            logits = model(model_input)
        next_token_logits = logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        token_ids = torch.cat((token_ids, next_token_id), dim=1)
        
    return token_ids

prompt = "There were many requests from my little correspondents"
prompt_ids = torch.tensor(
    tokenizer.encode(prompt), dtype=torch.long
).unsqueeze(0).to(device)

generated_ids = generate_text(
    model=model,
    token_ids=prompt_ids,
    max_new_tokens=5,
    context_size=GPT_CONFIG_124M["context_length"],
)

generated_text = tokenizer.decode(generated_ids.squeeze(0).tolist())
print("Generated text:", generated_text)

Generated text: There were many requests from my little correspondentsiors gigg Games Wah bets
